# Download Images

In [12]:
# import os

# zip_url = "https://storage.googleapis.com/bean-flowering/images.zip"
# zip_filename = "images.zip"
# images_dir = "images"

# # Download the zip file
# print(f"Downloading {zip_filename} from {zip_url}...")
# os.system(f"wget -O {zip_filename} {zip_url}")

# # Unzip the file
# print(f"Unzipping {zip_filename}...")
# os.system(f"unzip -q {zip_filename}")

# # Remove the zip file after extraction
# os.system(f"rm {zip_filename}")

# print(f"Images downloaded and extracted to ./{images_dir}/")

# Part 1: Config & Imports



Set project paths, training parameters, category mapping, and reusable helpers for the YOLO workflow.

In [13]:
# !pip install -q ultralytics

In [14]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [15]:
import ast
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

PROJECT_DIR = Path.cwd()

TRAIN_CSV = PROJECT_DIR / "Train.csv"

TEST_CSV = PROJECT_DIR / "Test.csv"

IMAGES_DIR = PROJECT_DIR / "images"


YOLO_DATASET_DIR = PROJECT_DIR / "yolo_dataset"

YOLO_IMAGES_TRAIN = YOLO_DATASET_DIR / "images" / "train"

YOLO_IMAGES_VAL = YOLO_DATASET_DIR / "images" / "val"

YOLO_LABELS_TRAIN = YOLO_DATASET_DIR / "labels" / "train"

YOLO_LABELS_VAL = YOLO_DATASET_DIR / "labels" / "val"

DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"


RUNS_DIR = PROJECT_DIR / "runs"

SUBMISSION_PATH = PROJECT_DIR / "submission.csv"


MODEL_WEIGHTS = (
    "yolo11l-seg.pt" # <- this is an upgrade "yolo11n-seg.pt"
)

EPOCHS = 100

IMAGE_SIZE = 1280

BATCH_SIZE = 4

VAL_SIZE = 0.15

RANDOM_STATE = 42

CONF_THRESHOLD = 0.20

IOU_THRESHOLD = 0.45

N_JOBS_DATA_PREP = 4


CLASS_CONF_THRESHOLDS = {
    "Plant_bean":    0.25,
    "Flower_open":   0.25,
    "Flower_closed": 0.18,   # starter model recall was only 0.173 for this class
}

CLASS_NAMES = ["Plant_bean", "Flower_open", "Flower_closed"]

CLASS_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

TRAIN_REQUIRED_COLS = {"Image_ID", "image_width", "image_height", "bbox", "points", "class"}
TEST_REQUIRED_COLS = {"Image_ID", "image_width", "image_height"}


for path in [TRAIN_CSV, TEST_CSV, IMAGES_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")


print("Config ready")

print(f"Project dir: {PROJECT_DIR}")

print(f"Train CSV:   {TRAIN_CSV}")

print(f"Test CSV:    {TEST_CSV}")

print(f"Images dir:  {IMAGES_DIR}")

print(f"Model:       {MODEL_WEIGHTS}")

print(f"Epochs:      {EPOCHS}")

print(f"Image size:  {IMAGE_SIZE}")

print(f"Batch size:  {BATCH_SIZE}")

print(f"Data prep workers: {N_JOBS_DATA_PREP}")

print(f"Expected Train.csv columns: {sorted(TRAIN_REQUIRED_COLS)}")

print(f"Expected Test.csv columns:  {sorted(TEST_REQUIRED_COLS)}")


Config ready
Project dir: /content
Train CSV:   /content/Train.csv
Test CSV:    /content/Test.csv
Images dir:  /content/images
Model:       yolo11l-seg.pt
Epochs:      100
Image size:  1280
Batch size:  4
Data prep workers: 4
Expected Train.csv columns: ['Image_ID', 'bbox', 'class', 'image_height', 'image_width', 'points']
Expected Test.csv columns:  ['Image_ID', 'image_height', 'image_width']


# Part 2: Data Preparation

Create YOLO train/validation folders, convert `Train.csv` bounding boxes into YOLO label files, and write the dataset YAML file.

Expected input format:
- `Train.csv`: `Image_ID`, `image_width`, `image_height`, `bbox`, `points`, `class`
- `Test.csv`: `Image_ID`, `image_width`, `image_height`

In [16]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Validate columns
for required, df, name in [
    (TRAIN_REQUIRED_COLS, train_df, "Train.csv"),
    (TEST_REQUIRED_COLS,  test_df,  "Test.csv"),
]:
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {sorted(missing)}")

invalid_classes = sorted(set(train_df["class"].dropna()) - set(CLASS_NAMES))
if invalid_classes:
    raise ValueError(f"Train.csv unsupported classes: {invalid_classes}")

# Print class distribution (useful for understanding imbalance)
print("Class distribution in Train.csv:")
print(train_df["class"].value_counts())


# ─── Helpers ──────────────────────────────────────────────────────────────────
def parse_bbox(value):
    if isinstance(value, str):
        parts = [p.strip() for p in value.strip().strip("[]").split(",")]
        if len(parts) != 4:
            raise ValueError(f"Invalid bbox: {value}")
        return [float(p) for p in parts]
    if isinstance(value, (list, tuple)) and len(value) == 4:
        return [float(v) for v in value]
    raise ValueError(f"Unsupported bbox: {value}")


def parse_points(value):
    if pd.isna(value):
        return []
    if isinstance(value, (list, tuple)):
        return [(float(p[0]), float(p[1])) for p in value if len(p) >= 2]
    s = str(value).strip()
    if not s:
        return []
    if s[0] in "[(":
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [(float(p[0]), float(p[1])) for p in parsed
                        if isinstance(p, (list, tuple)) and len(p) >= 2]
        except Exception:
            pass
    pts = []
    for token in s.split(";"):
        token = token.strip()
        pair = [v.strip() for v in token.split(",")]
        if len(pair) >= 2:
            try:
                pts.append((float(pair[0]), float(pair[1])))
            except ValueError:
                pass
    return pts


def normalize_polygon(points, w, h):
    return [(min(1.0, max(0.0, x / w)), min(1.0, max(0.0, y / h))) for x, y in points]


def bbox_to_polygon(x, y, w, h):
    return [(x, y), (x+w, y), (x+w, y+h), (x, y+h)]


# ─── Train/val split ─────────────────────────────────────────────────────────
unique_ids = train_df["Image_ID"].drop_duplicates().tolist()
train_ids, val_ids = train_test_split(
    unique_ids, test_size=VAL_SIZE, random_state=RANDOM_STATE, shuffle=True
)
train_ids, val_ids = set(train_ids), set(val_ids)

# Create directories
for folder in [YOLO_IMAGES_TRAIN, YOLO_IMAGES_VAL, YOLO_LABELS_TRAIN, YOLO_LABELS_VAL]:
    folder.mkdir(parents=True, exist_ok=True)


def link_or_copy_image(src, dst):
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def process_one_image(image_id, group, images_dir, labels_dir):
    image_path = IMAGES_DIR / f"{image_id}.jpg"
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")
    link_or_copy_image(image_path, images_dir / image_path.name)

    iw = float(group["image_width"].iloc[0])
    ih = float(group["image_height"].iloc[0])
    label_lines = []

    for _, row in group.iterrows():
        class_id = CLASS_TO_ID[row["class"]]
        x, y, w, h = parse_bbox(row["bbox"])
        poly = parse_points(row.get("points"))
        if len(poly) < 3:
            poly = bbox_to_polygon(x, y, w, h)
        norm_poly = normalize_polygon(poly, iw, ih)
        if len(norm_poly) < 3:
            continue
        coords = " ".join(f"{px:.6f} {py:.6f}" for px, py in norm_poly)
        label_lines.append(f"{class_id} {coords}")

    (labels_dir / f"{image_id}.txt").write_text("\n".join(label_lines) + "\n", encoding="utf-8")
    return image_id


def write_labels(split_name, split_ids, images_dir, labels_dir):
    split_rows = train_df[train_df["Image_ID"].isin(split_ids)].copy()
    grouped = list(split_rows.groupby("Image_ID", sort=False))
    processed = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(process_one_image)(img_id, grp, images_dir, labels_dir)
        for img_id, grp in grouped
    )
    print(f"{split_name}: prepared {len(processed)} images")


write_labels("train", train_ids, YOLO_IMAGES_TRAIN, YOLO_LABELS_TRAIN)
write_labels("val",   val_ids,   YOLO_IMAGES_VAL,   YOLO_LABELS_VAL)

# Write data.yaml
data_yaml = {
    "path":  str(YOLO_DATASET_DIR),
    "train": str(YOLO_IMAGES_TRAIN.relative_to(YOLO_DATASET_DIR)),
    "val":   str(YOLO_IMAGES_VAL.relative_to(YOLO_DATASET_DIR)),
    "names": CLASS_NAMES,
}
DATA_YAML_PATH.write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")
print("\nYOLO dataset ready")
print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | YAML: {DATA_YAML_PATH}")

Class distribution in Train.csv:
class
Flower_closed    6003
Flower_open      4085
Plant_bean       2054
Name: count, dtype: int64


NameError: name 'N_JOBS' is not defined

# Part 3: Model Training



Train the YOLO model for 10 epochs with image size 640 and batch size 8.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data        = str(DATA_YAML_PATH),
    epochs      = EPOCHS,
    patience    = 20,          # early stop if no improvement for 20 epochs
    imgsz       = IMAGE_SIZE,
    batch       = BATCH_SIZE,
    project     = str(RUNS_DIR),
    name        = "yolo_train",
    exist_ok    = True,
    seed        = RANDOM_STATE,

    # ── Optimizer / LR ──────────────────────────────────────────────────────
    optimizer   = "AdamW",
    lr0         = 0.001,
    lrf         = 0.01,
    cos_lr      = True,        # cosine decay — smoother convergence
    warmup_epochs = 3,

    # ── Augmentation ────────────────────────────────────────────────────────
    mosaic      = 1.0,         # always use mosaic (great for small object recall)
    copy_paste  = 0.3,         # copy-paste augmentation for small dense objects
    mixup       = 0.1,
    degrees     = 10.0,        # slight rotation (plants can be at angles)
    translate   = 0.1,
    scale       = 0.5,
    flipud      = 0.3,         # vertical flips useful for overhead crop images
    fliplr      = 0.5,
    hsv_h       = 0.015,
    hsv_s       = 0.7,
    hsv_v       = 0.4,
    erasing     = 0.4,

    # ── Loss weights ────────────────────────────────────────────────────────
    box         = 7.5,
    cls         = 1.0,         # higher than default (0.5) → better class accuracy
    dfl         = 1.5,

    # ── Misc ────────────────────────────────────────────────────────────────
    amp         = True,        # mixed precision — faster + less VRAM
    workers     = 4,
    plots       = True,
)

best_weights_path = Path(model.trainer.best)
print(f"\n✅ Training complete. Best weights: {best_weights_path}")

# Part 4: Inference



Run inference on test images using the best trained weights and collect per-detection predictions.

In [ ]:
# Load best checkpoint
best_model = YOLO(best_weights_path)

records = []

for _, row in test_df.iterrows():
    image_id   = str(row["Image_ID"])
    img_w      = int(row["image_width"])
    img_h      = int(row["image_height"])
    image_path = IMAGES_DIR / f"{image_id}.jpg"

    if not image_path.exists():
        records.append({
            "Image_ID": image_id,
            "class": "__background__",
            "confidence": 0.0,
            "ymin": 0, "xmin": 0, "ymax": 0, "xmax": 0,
        })
        continue

    # ── Run inference with TTA ─────────────────────────────────────────────
    results = best_model.predict(
        source       = str(image_path),
        imgsz        = IMAGE_SIZE,
        conf         = min(CLASS_CONF_THRESHOLDS.values()),  # global lower bound
        iou          = IOU_THRESHOLD,
        augment      = True,    # ← TTA
        retina_masks = True,    # higher quality masks → better bbox bounds
        verbose      = False,
    )

    result = results[0]
    detected = False

    if result.boxes is not None and len(result.boxes) > 0:
        boxes  = result.boxes.xyxy.cpu().numpy()    # x1,y1,x2,y2 (pixel)
        confs  = result.boxes.conf.cpu().numpy()
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)

        # Scale back to original image dimensions
        # (YOLO resizes internally; boxes are already in original pixel coords
        #  when using retina_masks=True on the original image)

        for bbox, conf, cls_id in zip(boxes, confs, cls_ids):
            class_name = ID_TO_CLASS[cls_id]

            # Apply per-class confidence filter
            if conf < CLASS_CONF_THRESHOLDS[class_name]:
                continue

            x1, y1, x2, y2 = bbox

            # Clip to image bounds
            x1 = max(0, min(img_w, x1))
            y1 = max(0, min(img_h, y1))
            x2 = max(0, min(img_w, x2))
            y2 = max(0, min(img_h, y2))

            records.append({
                "Image_ID": image_id,
                "class":    class_name,
                "confidence": float(conf),
                "ymin": int(round(y1)),
                "xmin": int(round(x1)),
                "ymax": int(round(y2)),
                "xmax": int(round(x2)),
            })
            detected = True

    if not detected:
        records.append({
            "Image_ID": image_id,
            "class": "__background__",
            "confidence": 0.0,
            "ymin": 0, "xmin": 0, "ymax": 0, "xmax": 0,
        })

predictions_df = pd.DataFrame(records)
print(f"Total predictions: {len(predictions_df)}")
print(predictions_df["class"].value_counts())

# Part 5: Submission Creation


Create the final submission file with Zindi columns `Image_ID`, `class`, `confidence`, `ymin`, `xmin`, `ymax`, and `xmax`.

In [ ]:
submission_columns = ["Image_ID", "class", "confidence", "ymin", "xmin", "ymax", "xmax"]

if predictions_df.empty:
    submission_df = pd.DataFrame(columns=submission_columns)
else:
    submission_df = predictions_df[submission_columns].copy()
    submission_df = submission_df.sort_values(
        ["Image_ID", "confidence"], ascending=[True, False]
    ).reset_index(drop=True)

# Ensure every Test.csv Image_ID is present
test_ids      = set(test_df["Image_ID"].astype(str).unique())
submission_ids = set(submission_df["Image_ID"].astype(str).unique()) if not submission_df.empty else set()
missing_ids   = sorted(test_ids - submission_ids)

if missing_ids:
    placeholder_df = pd.DataFrame({
        "Image_ID":   missing_ids,
        "class":      ["__background__"] * len(missing_ids),
        "confidence": [0.0] * len(missing_ids),
        "ymin": [0] * len(missing_ids),
        "xmin": [0] * len(missing_ids),
        "ymax": [0] * len(missing_ids),
        "xmax": [0] * len(missing_ids),
    })
    submission_df = pd.concat([submission_df, placeholder_df], ignore_index=True)
    print(f"Added {len(missing_ids)} placeholder rows for images with no detections.")

# Cast coordinates to int
coord_cols = ["xmin", "ymin", "xmax", "ymax"]
for c in coord_cols:
    submission_df[c] = pd.to_numeric(submission_df[c], errors="coerce")
invalid = int(submission_df[coord_cols].isna().any(axis=1).sum())
if invalid > 0:
    raise ValueError(f"{invalid} rows with invalid bbox coords.")
submission_df[coord_cols] = submission_df[coord_cols].round().astype(int)

submission_df = submission_df.sort_values(
    ["Image_ID", "confidence"], ascending=[True, False]
).reset_index(drop=True)

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"\n✅ Submission saved: {SUBMISSION_PATH}")
print(f"   Rows: {len(submission_df)}")
print(f"   Unique Image_IDs: {submission_df['Image_ID'].nunique()} / {len(test_ids)}")
print("\nClass breakdown:")
print(submission_df["class"].value_counts())

submission_df.head(10)

In [ ]:
# Run official YOLO val to see mAP50 per class
val_results = best_model.val(
    data    = str(DATA_YAML_PATH),
    imgsz   = IMAGE_SIZE,
    conf    = 0.001,          # low conf for full PR curve
    iou     = IOU_THRESHOLD,
    split   = "val",
    plots   = True,
)

print("\n=== Validation Results ===")
print(f"mAP50 (all classes): {val_results.box.map50:.4f}")
print(f"Precision:           {val_results.box.mp:.4f}")
print(f"Recall:              {val_results.box.mr:.4f}")

# Per-class
for i, name in enumerate(CLASS_NAMES):
    ap = val_results.box.ap50[i]
    print(f"  {name:20s} AP50: {ap:.4f}")